# Control Charts for Multi-Agent Systems (Helm, Priebe & Duderstadt 2026)

**Goal**: After this session, you will be able to implement, from memory, the machinery of statistical process control for multi-agent AI deployments: a classical-MDS "iso-mirror" trajectory, an adaptive Shewhart control chart, a sleeper agent's sigmoid defection schedule — and reproduce the paper's two headline results: slow defection evades adaptive monitoring (Theorem 1), and no chart is simultaneously adaptive and secure (Theorem 2).
**Time**: 1 hour.

## What and Why

Deployed multi-agent systems change over time — agents learn from the environment and from each other — so you cannot freeze a behavioral baseline and alarm on any deviation. This paper imports *control charts* (the factory-floor tool for "is this process still in control?") and shows that the fix, adaptive limits recomputed on a rolling window, opens a security hole: an adversary that defects **slowly** gets its drift absorbed into the baseline and is never flagged. The punchline is a genuine impossibility: **either agents can't learn, or the system is susceptible to adversaries.** This is the process-control complement to the AI-control line you implemented in `practice/05`.

## Key Formulas

**Adaptive Shewhart chart** on a monitored scalar $\phi^{(t)}$, rolling window $\mathcal{W}^{(t)} = \{t-w, \dots, t-1\}$:

$$\hat\mu^{(t)} = \frac{1}{w}\sum_{\tau \in \mathcal{W}^{(t)}} \phi^{(\tau)}, \qquad \hat\sigma^{(t)} = \operatorname{std}\big(\phi^{(\tau)}: \tau \in \mathcal{W}^{(t)}\big)$$

$$\text{alarm at } t \iff \phi^{(t)} \notin \big[\hat\mu^{(t)} - k\hat\sigma^{(t)},\; \hat\mu^{(t)} + k\hat\sigma^{(t)}\big]$$

**Sleeper defection schedule** (sigmoid ramp, normalized to hit exactly $0$ and $p_{\max}$ at the ends):

$$P_{\text{defect}}(t) = \begin{cases} 0 & t < t^* \\[2pt] p_{\max}\,\dfrac{\sigma\!\big(\kappa\,(\tfrac{2(t-t^*)}{s} - 1)\big) - \sigma(-\kappa)}{\sigma(\kappa) - \sigma(-\kappa)} & t^* \le t < t^*+s \\[2pt] p_{\max} & t \ge t^*+s \end{cases}$$

**Where:**
- $\phi^{(t)}$ — the iso-mirror coordinate at time $t$: the 1-d classical-MDS embedding of pairwise distances between system-state snapshots
- $w$ — rolling-window length; $k$ — chart sensitivity in sigmas ($k=3$ is the classic "3-sigma rule")
- $\sigma(\cdot)$ — the logistic function; $t^*$ — sleeper activation time; $s$ — ramp duration (small $s$ = fast attack, large $s$ = slow attack); $p_{\max}$ — asymptotic defection probability; $\kappa$ — ramp steepness
- A defecting agent answers wrong with probability $P_{\text{defect}}(t)$, so its mean embedding is displaced **proportionally to** $P_{\text{defect}}(t)$ — same final displacement whether the ramp was fast or slow. Detection can therefore only depend on *speed*, which is exactly what Theorem 1 exploits.

## Component Breakdown

- **Part 1 — Iso-mirror**: pairwise distances between state snapshots $\to$ double-centering $\to$ top eigenpair $\to$ 1-d trajectory $\phi^{(t)}$
- **Part 2 — Adaptive chart**: rolling $\hat\mu \pm k\hat\sigma$ limits + alarm mask; calibrated against the analytic Gaussian false-alarm rate
- **Part 3 (Recall)** — distribution-free threshold calibration via quantiles (from `practice/05`)
- **Part 4 — Defection schedule**: the normalized sigmoid ramp
- **Part 5 — Theorem 1, empirically**: Monte-Carlo exceedance rates; fast attack lights up the chart, slow attack is statistically invisible
- **Part 6 — Theorem 2, empirically**: static vs adaptive charts on legitimate-learning vs slow-attack scenarios — each chart fails exactly one cell of the 2×2

## References

- Helm, Priebe & Duderstadt, *Control Charts for Multi-agent Systems*, [arXiv:2605.11135](https://arxiv.org/abs/2605.11135)
- Iso-mirror lineage: Athreya et al., *Euclidean mirrors and dynamics prediction* (classical MDS on time-indexed model states)
- Sibling problem: `practice/05-Multi-Agent-Control-Collusion` (AI-control safety metric, collusion)


In [ ]:
import math

import torch
from jaxtyping import Bool, Float, Int
from torch import Tensor

torch.manual_seed(0)

# ---- Deployment constants (paper's experiment, scaled down for CPU) ----
N_AGENTS, D_AGENT = 8, 16
D = N_AGENTS * D_AGENT   # concatenated system-state dim
T = 600                  # deployment length (timesteps); paper uses 1000
W = 100                  # rolling window for adaptive limits (paper: 200)
T_STAR = 250             # sleeper activation time (paper: 400)
BURN_IN = 200            # clean period the *static* chart freezes its limits on
S_FAST, S_SLOW = 15, 300  # defection ramp durations (paper: 10 vs 800)
P_MAX = 0.5              # asymptotic defection probability
KAPPA = 6.0              # ramp steepness
NOISE_STD = 1.0          # benign per-coordinate jitter of agent embeddings
SHIFT_NORM = 9.0         # state displacement at full defection / full drift

steps = torch.arange(T)  # shared timestep index


def simulate(
    defect_prob: Float[Tensor, "T"] | None = None,
    legit_drift: bool = False,
    seed: int = 0,
) -> Float[Tensor, "T D"]:
    """One deployment trajectory of system state (all provided — no TODO here).

    Returns E of shape (T, D): at each timestep, the concatenation of all
    N_AGENTS agents' response embeddings.
    # same shape/role as stacking the paper's RAG agents' answer embeddings
    # at each step; here: resting state + benign jitter + optional shifts.

    - defect_prob: (T,) sleeper defection probability. A defecting agent
      answers wrong with prob p, so its MEAN embedding is displaced
      proportionally to p. Only agent 0's block moves.
    - legit_drift: the environment changed and ALL agents legitimately
      re-learn — the whole state shifts along a shared direction
      (linear ramp over S_SLOW steps starting at T_STAR).
    """
    g = torch.Generator().manual_seed(seed)
    base = torch.randn(D, generator=g)
    E = base + NOISE_STD * torch.randn(T, D, generator=g)
    if legit_drift:
        gdir = torch.randn(D, generator=g)
        gdir = gdir / gdir.norm()
        ramp = ((steps.float() - T_STAR) / S_SLOW).clamp(0.0, 1.0)
        E = E + SHIFT_NORM * ramp[:, None] * gdir
    if defect_prob is not None:
        u = torch.zeros(D)
        u[:D_AGENT] = torch.randn(D_AGENT, generator=g)
        u = u / u.norm()
        E = E + SHIFT_NORM * (defect_prob / P_MAX)[:, None] * u
    return E


print(f"system dim D = {D}, deployment length T = {T}, window w = {W}")


## Part 1: The Iso-Mirror — a 1-d Trajectory of System State

The chart needs a scalar to monitor. The paper's choice: compute pairwise distances between system-state snapshots across time, then embed them into 1-d with **classical MDS** — the resulting trajectory $\phi^{(t)}$ is the "iso-mirror". Implement `iso_mirror`: squared distance matrix $\to$ double-centering $\to$ top eigenpair $\to$ coordinates.

**Insight (idea-novel step)**: double-centering the squared-distance matrix — $B = -\tfrac12 J D^{(2)} J$ with $J = I - \tfrac1T \mathbf{1}\mathbf{1}^\top$ — converts squared distances into a Gram matrix of inner products about the centroid; the top eigenvector scaled by $\sqrt{\lambda}$ is then the best 1-d embedding, exact when the points truly lie on a line. *In one line, explain why centering rows and columns of $-\tfrac12 D^{(2)}$ recovers inner products.*

**Predict before you code**: the test embeds 50 points that lie exactly on a line in $\mathbb{R}^{32}$. How large will the worst-case error between $|\phi_i - \phi_j|$ and the true $|x_i - x_j|$ be? Commit an upper bound as the first line of the stub.


In [ ]:
predicted_max_dist_err = ...  # commit a number before you code: upper bound on max |cdist(phi) - cdist(x)|


def iso_mirror(E: Float[Tensor, "T D"]) -> Float[Tensor, "T"]:
    """1-d classical-MDS embedding of the trajectory's pairwise distances."""
    # TODO: Implement
    pass


In [ ]:
# --- Part 1 Validation ---
torch.manual_seed(42)
x = torch.sort(torch.rand(50) * 10).values           # 1-d latent positions
u1 = torch.randn(32)
u1 = u1 / u1.norm()
E_line = x[:, None] * u1 + 3.0                        # a line embedded in R^32

phi_line = iso_mirror(E_line)

dist_err = (torch.cdist(phi_line[:, None], phi_line[:, None])
            - torch.cdist(x[:, None], x[:, None])).abs().max()
print(f"  You predicted max distance error < {predicted_max_dist_err}; actual {dist_err:.2e}"
      f"  (a large gap is the most valuable moment of the session)")

assert phi_line.shape == (50,), f"Shape: expected (50,), got {tuple(phi_line.shape)}"
print(f"  Shape: {tuple(phi_line.shape)} -- correct")
print(f"  Range: [{phi_line.min():.3f}, {phi_line.max():.3f}], mean {phi_line.mean():.4f}, std {phi_line.std():.3f}")

assert phi_line.mean().abs() < 1e-3, (
    "iso-mirror coordinates must be centered: double-centering puts the constant vector "
    "in B's null space, so every eigenvector with nonzero eigenvalue has zero mean")
print("  Centered -- correct")

assert dist_err < 1e-2, (
    f"Pairwise distances not preserved (max err {dist_err:.2e}, threshold 1e-2). "
    "For points on a line CMDS is an exact isometry -- check the -1/2 factor, "
    "that you squared the distances, and the sqrt(eigenvalue) scaling.")
print(f"  Isometry on a 1-d manifold -- correct (max err {dist_err:.2e})")

print("\nPart 1 complete.")


## Part 2: The Adaptive Shewhart Chart

Implement the chart itself: at each $t \ge w$, estimate $\hat\mu, \hat\sigma$ from the window $\phi^{(t-w)} \dots \phi^{(t-1)}$ (exclusive of $t$) and alarm iff $|\phi^{(t)} - \hat\mu| > k\hat\sigma$. Entries before $t = w$ are `False` (no limits exist yet). This *is* the paper's monitoring procedure — everything after this is attack and evaluation.

**Predict before you code**: on iid $\mathcal{N}(0,1)$ data with $k=2$, what fraction of timesteps will alarm? (This is a number you know from the Gaussian two-sided tail.) Commit it as the first line of the stub.


In [ ]:
predicted_far_k2 = ...  # commit: expected false-alarm rate on iid N(0,1) at k=2


def adaptive_chart(phi: Float[Tensor, "T"], w: int, k: float) -> Bool[Tensor, "T"]:
    """Alarm mask. For t >= w: alarm iff |phi[t] - mean(phi[t-w:t])| > k * std(phi[t-w:t]).
    alarms[:w] = False."""
    # TODO: Implement
    pass


In [ ]:
# --- Part 2 Validation ---
torch.manual_seed(42)
phi_iid = torch.randn(8000)
alarms_iid = adaptive_chart(phi_iid, 200, 2.0)

far = alarms_iid[200:].float().mean().item()
analytic = 2 * (1 - 0.5 * (1 + math.erf(2.0 / math.sqrt(2.0))))  # 2*Phi(-2)
print(f"  You predicted false-alarm rate ~{predicted_far_k2}; actual {far:.4f}"
      f"  (analytic 2*Phi(-2) = {analytic:.4f})")

assert alarms_iid.dtype == torch.bool and alarms_iid.shape == (8000,), (
    f"expected bool mask of shape (8000,), got {alarms_iid.dtype} {tuple(alarms_iid.shape)}")
assert not alarms_iid[:200].any(), "no limits exist before one full window of history -- alarms[:w] must be False"
print("  Shape/dtype and warm-up masking -- correct")

assert abs(far - analytic) < 0.02, (
    f"False-alarm rate {far:.4f} should match the nominal Gaussian rate {analytic:.4f} within 0.02. "
    "0.0 means your inequality never fires (check > vs >=, and k*std); "
    ">>0.05 usually means the window includes phi[t] itself or is not rolling.")
print(f"  Nominal false-alarm rate at k=2 -- correct ({far:.4f})")

# Adaptivity property: a slow linear trend must NOT alarm (the window tracks it).
trend = torch.arange(4000).float() * 0.005 + torch.randn(4000)   # 0.5 sigma drift per 100 steps
far_trend = adaptive_chart(trend, 200, 3.0)[200:].float().mean().item()
print(f"  Alarm rate on a slow linear trend at k=3: {far_trend:.4f}")
assert far_trend < 0.02, (
    "an ADAPTIVE chart absorbs slow trends into its rolling baseline; if this alarms, "
    "your mu/sigma are probably computed once instead of per-timestep")
print("  Tracks slow trends without alarming -- correct (this exact property becomes the vulnerability in Part 5)")

print("\nPart 2 complete.")


## Part 3: Threshold Calibration by Quantile (Recall)

The $k\sigma$ rule assumes Gaussian residuals. The distribution-free alternative — pick the threshold as an empirical quantile of clean scores so that exactly a target fraction exceeds it — is the same operation as the audit-budget thresholding you implemented in `practice/05-Multi-Agent-Control-Collusion`. Reproduce it here from memory — your old worked solution is in that problem's `_solutions/` folder; opening it ends the rep, so only consult it after you've produced or genuinely failed an attempt.


In [ ]:
def calibrate_threshold(scores, target_far):
    # Implement from memory: threshold such that P(clean score > threshold) = target_far
    pass


In [ ]:
# --- Part 3 Validation ---
torch.manual_seed(42)
clean_scores = torch.randn(4000).abs()

thr = calibrate_threshold(clean_scores, 0.05)
emp = (clean_scores > thr).float().mean().item()
print(f"  threshold at target_far=0.05: {thr:.4f}  (|N(0,1)| 95th percentile is 1.96)")
print(f"  empirical exceedance on clean scores: {emp:.4f}")

assert isinstance(thr, float), f"return a plain float, got {type(thr)}"
assert abs(emp - 0.05) < 0.01, (
    "the whole point of a quantile threshold: exactly target_far of clean scores exceed it. "
    "Off by a lot? Check the quantile direction (1 - target_far, not target_far).")
assert abs(thr - 1.96) < 0.08, f"|N(0,1)| 95th percentile is 1.96, got {thr:.3f}"
assert calibrate_threshold(clean_scores, 0.01) > calibrate_threshold(clean_scores, 0.10), (
    "rarer alarms require a higher threshold -- monotonicity in target_far is inverted")
print("  Quantile calibration -- correct")

print("\nPart 3 complete.")


## Part 4: The Sleeper's Defection Schedule

Implement the sigmoid ramp from the Key Formula: zero before $t^*$, a normalized logistic ramp over $[t^*, t^*+s)$, then flat at $p_{\max}$. The normalization — subtract $\sigma(-\kappa)$, divide by $\sigma(\kappa) - \sigma(-\kappa)$ — makes the ramp start at exactly $0$ and end at exactly $p_{\max}$ instead of the raw sigmoid's asymptotes. Same function generates the fast attack ($s = 15$) and the slow attack ($s = 300$).

**Predict before you code**: what is $P_{\text{defect}}$ exactly halfway up the ramp, at $t = t^* + s/2$? (Think about the symmetry $\sigma(z) + \sigma(-z) = 1$.) Commit it as the first line of the stub.


In [ ]:
predicted_mid_prob = ...  # commit: P_defect at t = t_star + s/2, for p_max = 0.5


def defection_schedule(
    t: Int[Tensor, "T"], t_star: int, s: int, p_max: float, kappa: float = 6.0
) -> Float[Tensor, "T"]:
    """Sleeper defection probability at each timestep (see Key Formula)."""
    # TODO: Implement
    pass


In [ ]:
# --- Part 4 Validation ---
p_slow = defection_schedule(steps, T_STAR, S_SLOW, P_MAX, KAPPA)
p_fast = defection_schedule(steps, T_STAR, S_FAST, P_MAX, KAPPA)

mid = p_slow[T_STAR + S_SLOW // 2].item()
print(f"  You predicted midpoint prob {predicted_mid_prob}; actual {mid:.4f}")

assert p_slow.shape == (T,), f"expected shape ({T},), got {tuple(p_slow.shape)}"
assert torch.allclose(p_slow[:T_STAR], torch.zeros(T_STAR)), "the sleeper is dormant before t_star"
assert p_slow[T_STAR].abs() < 1e-6, (
    "the ramp must start at exactly 0 -- that is what subtracting sigma(-kappa) is for")
assert torch.allclose(p_slow[T_STAR + S_SLOW:], torch.full((T - T_STAR - S_SLOW,), P_MAX)), (
    "after the ramp the schedule sits exactly at p_max")
assert abs(mid - P_MAX / 2) < 1e-4, (
    "sigmoid symmetry sigma(z)+sigma(-z)=1 makes the normalized ramp hit exactly p_max/2 at midpoint")
assert (p_slow.diff() >= -1e-7).all(), "defection probability must be monotone non-decreasing"
print("  Endpoints, plateau, midpoint symmetry, monotonicity -- correct")

print(f"  fast ramp reaches p_max at t = {int((p_fast >= P_MAX - 1e-6).nonzero()[0])}, "
      f"slow at t = {int((p_slow >= P_MAX - 1e-6).nonzero()[0])} "
      f"-- same destination, different speed. That difference is everything (Part 5).")

print("\nPart 4 complete.")


## Part 5: Theorem 1, Empirically — Slow Defection Evades the Chart

Compose your pieces into the paper's detectability experiment. Implement `detect(E, k)`: iso-mirror the trajectory, run the adaptive chart (window `W`), and return the **exceedance rate** — the fraction of post-activation timesteps ($t \ge$ `T_STAR`) that alarm. The validation runs a Monte-Carlo over deployments for three scenarios: no adversary, fast attack, slow attack. Both attacks end at the *same* total displacement.

**Predict before you code**: at $k=3$, what exceedance rate do you expect for the fast attack, and for the slow attack? Commit both as the first lines of the stub. (Baseline nominal rate at $k=3$ is $2\Phi(-3) \approx 0.003$.)


In [ ]:
predicted_fast_rate = ...  # commit: mean exceedance rate for the FAST attack at k=3
predicted_slow_rate = ...  # commit: mean exceedance rate for the SLOW attack at k=3


def detect(E, k):
    # Implement from memory: compose Part 1 + Part 2.
    # Fraction of timesteps t >= T_STAR where the adaptive chart (window W) alarms.
    pass


In [ ]:
# --- Part 5 Validation ---
R = 30
k3 = 3.0
profiles = {"none": None, "fast": p_fast, "slow": p_slow}
per_rep = {name: [detect(simulate(defect_prob=prof, seed=r), k3) for r in range(R)]
           for name, prof in profiles.items()}
rates = {n: sum(v) / R for n, v in per_rep.items()}
# a deployment counts as "flagged" if it produced >= 5 alarms after t*
flagged = {n: sum(x >= 4.5 / (T - T_STAR) for x in v) / R for n, v in per_rep.items()}

print(f"  You predicted fast ~{predicted_fast_rate}, slow ~{predicted_slow_rate}"
      f"  (a large gap is the most valuable moment of the session)")
print(f"  over {R} deployments @ k=3:")
print( "                  mean exceedance   deployments flagged (>=5 alarms)")
for n in ("none", "fast", "slow"):
    print(f"    {n:<12}  {rates[n]:.4f}            {flagged[n]:.2f}")

assert rates["fast"] > 0.02 and flagged["fast"] >= 0.9, (
    f"fast attack: rate {rates['fast']:.4f}, flagged {flagged['fast']:.2f} -- expected a burst of "
    "alarms in EVERY deployment: the jump sits outside limits for the ramp plus ~W/(1+k^2) steps, "
    "until the shifted points inflate the window's sigma-hat and swallow the deviation")
assert flagged["none"] <= 0.1, (
    f"no-adversary deployments should almost never reach 5 alarms (nominal rate 2*Phi(-3)=0.003), "
    f"got flagged={flagged['none']:.2f}")
assert abs(rates["slow"] - rates["none"]) < 0.02 and flagged["slow"] <= 0.3, (
    f"slow attack (rate {rates['slow']:.4f}, flagged {flagged['slow']:.2f}) should look like "
    f"baseline (rate {rates['none']:.4f}): the rolling baseline absorbs each window's small "
    "shift while the displacement accumulates -- Lemma 1 / Theorem 1")
print("  Theorem 1 reproduced: same total displacement, detection depends only on speed.")

# Sensitivity sweep (paper Fig. 4): at k=5 the slow attack is EXACTLY as invisible as no adversary
R2 = 10
sweep = {k_: {n: sum(detect(simulate(defect_prob=p, seed=r), k_) for r in range(R2)) / R2
              for n, p in profiles.items()} for k_ in (2.0, 5.0)}
print("\n  exceedance by sensitivity k:")
print( "      k    none    fast    slow")
for k_, r_ in [(2.0, sweep[2.0]), (3.0, rates), (5.0, sweep[5.0])]:
    print(f"    {k_:.0f}   {r_['none']:.4f}  {r_['fast']:.4f}  {r_['slow']:.4f}")
assert abs(sweep[5.0]["slow"] - sweep[5.0]["none"]) < 0.005, (
    "at k=5 the slow attack must be indistinguishable from no adversary (paper Fig. 4)")
print("  Note the k=2 row: the slow attack IS partially visible there -- a real attacker tunes "
      "the ramp to the window size and sensitivity actually in use.")

print("\nPart 5 complete.")


## Part 6: Theorem 2, Empirically — Adaptive XOR Secure

The obvious fix is to freeze the limits: estimate $\mu_0, \sigma_0$ once on a clean burn-in period and never update. Implement `static_chart` (limits from `phi[:burn_in]`, alarms only for $t \ge$ `burn_in`). The validation then fills the 2×2: {adaptive, static} charts × {legitimate system-wide learning, slow attack} scenarios.

**Predict before you code** (prose, no literal): which two cells of the 2×2 fail? I.e., where does each chart type break — and why is there no third column that wins both rows?


In [ ]:
def static_chart(phi, burn_in, k):
    # Implement from memory: limits frozen from phi[:burn_in]; alarms only for t >= burn_in
    pass


In [ ]:
# --- Part 6 Validation ---
torch.manual_seed(42)
sanity = static_chart(torch.randn(1000), 200, 3.0)
assert sanity.shape == (1000,) and sanity.dtype == torch.bool
assert not sanity[:200].any(), "static chart alarms only after its burn-in"
far_static = sanity[200:].float().mean().item()
assert far_static < 0.02, f"on iid data the static chart should alarm at ~2*Phi(-3)=0.003, got {far_static:.4f}"
print(f"  static chart sanity on iid data -- correct (rate {far_static:.4f})")

R3 = 12
post = lambda a: a[T_STAR:].float().mean().item()
table = {}
scenarios = {
    "legit learning": [simulate(legit_drift=True, seed=100 + r) for r in range(R3)],
    "slow attack":    [simulate(defect_prob=p_slow, seed=200 + r) for r in range(R3)],
}
for scen, sims in scenarios.items():
    phis = [iso_mirror(E) for E in sims]
    table[scen] = {
        "adaptive": sum(post(adaptive_chart(ph, W, 3.0)) for ph in phis) / R3,
        "static":   sum(post(static_chart(ph, BURN_IN, 3.0)) for ph in phis) / R3,
    }

print(f"\n  mean post-t* alarm rate @ k=3 over {R3} deployments:")
print( "                      adaptive   static")
for scen, row in table.items():
    print(f"    {scen:<16}  {row['adaptive']:.4f}     {row['static']:.4f}")

assert table["legit learning"]["adaptive"] < 0.05, (
    "adaptive charts must absorb legitimate re-learning without alarming (that is their purpose)")
assert table["legit learning"]["static"] > 0.30, (
    "static limits false-alarm on ANY legitimate shift exceeding k*sigma_0 -- "
    "with a 9-sigma benign drift, alarms approach certainty")
assert table["slow attack"]["static"] > 0.30, (
    "static limits DO catch slow drift (security) -- the displacement eventually leaves the frozen band")
assert table["slow attack"]["adaptive"] < 0.05, (
    "adaptive limits absorb the slow attack (Theorem 1) -- the vulnerability Part 5 measured")
print("\n  Theorem 2 reproduced: each chart fails exactly one row.")
print("  Adaptive = usable but insecure; static = secure but unusable. Pick your poison.")

print("\nPart 6 complete.")


## Session Debrief

Write your answers into the code cell below (typing is overt retrieval — far
stronger than answering "in your head"). Don't scroll up.

1. Write the adaptive control-limit formulas ($\hat\mu^{(t)}$, UCL/LCL) and the alarm rule from memory.
2. Why does slow defection evade an adaptive chart, in one sentence? (What property of the rolling baseline does the attacker exploit?)
3. State the learning–security tradeoff (Theorem 2) precisely.
4. What does double-centering do to a squared-distance matrix?

**Check yourself**: your worked solution is in `_solutions/` — open it (and the
paper) to grade your answers. Opening it ends the retrieval rep, so answer first.

**Challenge**: Close this notebook, open a blank one, and rewrite Part 5's full
pipeline (simulate → iso-mirror → chart → exceedance rate) from scratch without looking back.


In [ ]:
debrief = """
1.
2.
3.
4.
"""  # type your recall here before checking _solutions/


In [ ]:
# --- Session log: replace every "___" with a real value, then run ---
# (placeholders are STRINGS on purpose: a bare ___ is silently defined by IPython)
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "practice/06-Control-Charts-Multi-Agent",
    "date": datetime.date.today().isoformat(),
    "budget_min": 60,                                   # pre-filled
    "actual_min": "___",                                # how long it really took (number)
    "parts": [                                          # n + tier pre-filled; set outcome
        {"n": 1, "tier": 2, "outcome": "___"},          # unaided | hint | solution | failed
        {"n": 2, "tier": 2, "outcome": "___"},
        {"n": 3, "tier": 3, "outcome": "___"},
        {"n": 4, "tier": 2, "outcome": "___"},
        {"n": 5, "tier": 3, "outcome": "___"},
        {"n": 6, "tier": 3, "outcome": "___"},
    ],
    "difficulty": "___",                                # 1 (trivial) .. 5 (over my head)
    "stuck": "___",                                     # one phrase: where you got stuck
}
assert "___" not in json.dumps(record), "fill every ___ before running this cell"
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")


## Extension (Optional)

Try one of these variations:

- **CUSUM instead of Shewhart**: a CUSUM chart accumulates deviations ($C_t = \max(0, C_{t-1} + \phi^{(t)} - \hat\mu - \delta/2)$) and is the classical answer to slow drift. Implement it, re-run Part 5 — then explain why Theorem 2 still binds (hint: what happens on the legitimate-learning scenario?).
- **Detectability curve**: sweep the ramp duration $s \in \{10, 25, 50, 100, 200, 400\}$ and plot exceedance rate vs $s$ at $k=3$. Where is the knee relative to the window size $W$? That ratio is the attacker's design rule.
- **Vectorize the chart**: rewrite `adaptive_chart` without the Python loop using `phi.unfold(0, w, 1)` — verify with `torch.equal` against your loop version.


## Anki Cards

Add these to your deck:

**Card 1**
Front: You monitor a multi-agent deployment with a rolling-window k-sigma chart. An adversary wants to shift system state by a large δ without a single alarm. What's the strategy?
Back: Drift slower than the window absorbs

**Card 2**
Front: Control-charts-for-agents impossibility result (Helm et al. 2026): what can't a single monitoring procedure have both of?
Back: Adaptivity and security

**Card 3**
Front: You have a pairwise squared-distance matrix and want embedding coordinates (classical MDS). What two operations?
Back: Double-center, top eigenvectors × √λ
